In [1]:
# ========= 标准库 =========
import os
import time
import logging
from collections import Counter, defaultdict
from itertools import cycle
from typing import Dict, List
import sys

# ========= 科学计算 / 数据处理 =========
import joblib
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

# ========= 深度学习 & 机器学习 =========
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.datasets import mnist
from tensorflow.keras.initializers import HeNormal, glorot_uniform
from tensorflow.keras.layers import (
    Add,
    AveragePooling1D,
    BatchNormalization,
    Conv1D,
    Dense,
    Flatten,
    Input,
    MaxPooling1D,
    ZeroPadding1D,
    Activation,
)
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from keras.metrics import Precision, Recall  # 可替换为 tf.keras.metrics.*

# ========= 可视化 =========
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

label_col = "Attack_type"


In [3]:
df = pd.read_csv('./preprocessed_DNN.csv', low_memory=False)
#df = pd.read_csv('./downsampled_preprocessed_DNN.csv', low_memory=False)
feat_cols = list(df.columns)

feat_cols.remove(label_col)
feat_cols

skip_list = ["icmp.unused", "http.tls_port", "dns.qry.type", "mqtt.msg_decoded_as", "Attack_label"]
df.drop(skip_list, axis=1, inplace=True)

feat_cols = list(df.columns)
feat_cols.remove(label_col)

In [4]:
df

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,...,mqtt.conack.flags-1471198,mqtt.conack.flags-1471199,mqtt.conack.flags-1574358,mqtt.conack.flags-1574359,mqtt.protoname-0,mqtt.protoname-0.0,mqtt.protoname-MQTT,mqtt.topic-0,mqtt.topic-0.0,mqtt.topic-Temperature_and_Humidity
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.647490e+09,1594.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1,0.0,0.0,0.0,0.0,0.0,0.0,59.0,3.411509e+09,54649.0,1.0,...,False,False,False,False,True,False,False,True,False,False
2,0.0,0.0,0.0,0.0,0.0,0.0,5.0,1.099419e+09,31572.0,0.0,...,False,False,False,False,True,False,False,False,False,True
3,0.0,0.0,0.0,0.0,0.0,0.0,59.0,1.185254e+09,54569.0,0.0,...,False,False,False,False,True,False,False,True,False,False
4,0.0,0.0,0.0,0.0,0.0,0.0,56.0,1.795444e+09,36563.0,0.0,...,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1909666,0.0,0.0,0.0,0.0,0.0,0.0,115678289.0,1.254530e+09,30876.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1909667,0.0,0.0,0.0,0.0,0.0,0.0,56.0,1.594269e+09,11256.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1909668,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.213215e+09,59837.0,0.0,...,False,False,False,False,False,True,False,False,True,False
1909669,0.0,0.0,0.0,0.0,0.0,0.0,479.0,4.273690e+09,7664.0,0.0,...,False,False,False,False,False,True,False,False,True,False


In [5]:
df.Attack_type.value_counts()

Attack_type
Normal                   1363998
DDoS_UDP                  121567
DDoS_ICMP                  67939
SQL_injection              50826
DDoS_TCP                   50062
Vulnerability_scanner      50026
Password                   49933
DDoS_HTTP                  48544
Uploading                  36807
Backdoor                   24026
Port_Scanning              19977
XSS                        15066
Ransomware                  9689
Fingerprinting               853
MITM                         358
Name: count, dtype: int64

# Preprocessing (normalization and padding values)

In [6]:
# Z-score normalization
features = df.dtypes[df.dtypes != 'object'].index
df[features] = df[features].apply(
    lambda x: (x - x.mean()) / (x.std()))
# Fill empty values by 0
df = df.fillna(0)

In [7]:
label_encoder = LabelEncoder()
df['Attack_type'] = label_encoder.fit_transform(df['Attack_type'])


In [8]:
df.Attack_type.value_counts()

Attack_type
7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
10       9689
5         853
6         358
Name: count, dtype: int64

In [9]:
X_fs = df.drop([label_col], axis=1)
y = df[label_col]

In [10]:
X_fs

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,...,mqtt.conack.flags-1471198,mqtt.conack.flags-1471199,mqtt.conack.flags-1574358,mqtt.conack.flags-1574359,mqtt.protoname-0,mqtt.protoname-0.0,mqtt.protoname-MQTT,mqtt.topic-0,mqtt.topic-0.0,mqtt.topic-Temperature_and_Humidity
0,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,-0.061580,-1.361015,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,1.297649,1.229695,2.984767,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
2,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,-0.483885,0.102830,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,-1.427427,-0.632498,4.690833
3,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,-0.417746,1.225788,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
4,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,0.052423,0.346544,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1909666,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,0.502600,-0.364367,0.068844,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1909667,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,-0.102589,-0.889213,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1909668,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,0.374328,1.483028,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,-1.427422,1.581031,-0.213186,-1.427427,1.581031,-0.213182
1909669,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149150,1.961984,-1.064613,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,-1.427422,1.581031,-0.213186,-1.427427,1.581031,-0.213182


# Solve class-imbalance by SMOTE

In [11]:
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import pandas as pd

def hybrid_resample(X, y, upsample_targets=[5, 6], total_per_class=9689, random_state=42):
    # 创建 DataFrame
    df = pd.DataFrame(X)
    df['label'] = y  # 假设标签列为 'label'，可以根据实际情况修改
    
    # 1. 分离需要上采样的类和其他类
    df_upsample = df[df['label'].isin(upsample_targets)]
    df_other = df[~df['label'].isin(upsample_targets)]
    
    # 2. 对需要上采样的类分别上采样到 total_per_class
    df_list = []
    for cls in upsample_targets:
        df_cls = df_upsample[df_upsample['label'] == cls]
        df_cls_up = resample(
            df_cls,
            replace=True,  # 允许重复采样
            n_samples=total_per_class,
            random_state=random_state
        )
        df_list.append(df_cls_up)
    
    # 合并上采样的类别
    df_upsampled = pd.concat(df_list, axis=0)
    
    # 4. 合并上采样后的类别和未变化的其他类
    df_final = pd.concat([df_upsampled, df_other], axis=0).sample(frac=1.0, random_state=random_state)

    # 5. 拆分特征和标签
    X_balanced = df_final.drop('label', axis=1).values
    y_balanced = df_final['label'].values

    return X_balanced, y_balanced

# 使用示例
X_balanced, y_balanced = hybrid_resample(X_fs, y, upsample_targets=[5, 6], total_per_class=9689)

In [12]:
pd.Series(y).value_counts()

Attack_type
7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
10       9689
5         853
6         358
Name: count, dtype: int64

In [13]:
pd.Series(y_balanced).value_counts()

7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
6        9689
10       9689
5        9689
Name: count, dtype: int64

In [14]:
def _earth_movers_distance(p, q):
    """一维 EMD（L1 距离简化版，足够衡量类别分布差异）"""
    return np.sum(np.abs(p - q))

def partition_dataset(
    X, y,
    num_clients: int = 10,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    non_iid: bool = False,
    classes_per_client: int = 2,
    share_pct: float = 0.0,      # β：全局共享数据占训练集比例（0~1）
    share_fraction: float = 1.0, # α：每个客户端接收共享集的比例（0~1）
    random_state: int = 42,
):
    """
    改进版数据划分：
    ------------------------------------------------------------------
    • IID  : non_iid=False
    • Non-IID: non_iid=True 且 classes_per_client=k
    • 共享策略: 设置 share_pct>0 触发全局共享数据 G
    返回:
        client_data : {cid: (X_c, y_c)}
        test_data   : (X_test, y_test)
        val_data    : (X_val,  y_val)
        stats       : {
                        'label_hist': {cid: p_vec},
                        'emd':       {cid: emd_val},
                        'global_p':  overall_p
                       }
    """
    rng = np.random.default_rng(random_state)

    # ---------- 1) 先划分 Train / Val / Test ----------
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=val_ratio + test_ratio,
        stratify=y,
        random_state=random_state,
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=test_ratio / (val_ratio + test_ratio),
        stratify=y_temp,
        random_state=random_state,
    )

    # ---------- 2) 选出全局共享数据 G ----------
    if share_pct > 0:
        g_size   = int(len(X_train) * share_pct)
        g_indices = rng.choice(len(X_train), size=g_size, replace=False)
        X_G, y_G  = X_train[g_indices], y_train[g_indices]

        # 剩余数据供真正划分
        keep_mask         = np.ones(len(X_train), dtype=bool)
        keep_mask[g_indices] = False
        X_train, y_train  = X_train[keep_mask], y_train[keep_mask]
    else:
        X_G, y_G = np.empty((0, *X_train.shape[1:])), np.empty((0,), dtype=y.dtype)

    # ---------- 3) 本地数据按 IID / Non-IID 划分 ----------
    if not non_iid:
        skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=random_state)
        client_indices = [idx for _, idx in skf.split(X_train, y_train)]
    else:
        # ===== Non-IID：每端 classes_per_client 个类别 =====
        buckets = defaultdict(list)
        for idx, lbl in enumerate(y_train):
            buckets[lbl].append(idx)
        for lbl in buckets:
            rng.shuffle(buckets[lbl])

        all_labels = np.array(sorted(buckets.keys()))
        client_classes = {cid: set() for cid in range(num_clients)}

        # 覆盖所有类别
        for lbl in all_labels:
            client_classes[rng.integers(num_clients)].add(lbl)
        # 补足到指定类别数
        for cid in range(num_clients):
            need = classes_per_client - len(client_classes[cid])
            if need > 0:
                extras = rng.choice(
                    all_labels[~np.isin(all_labels, list(client_classes[cid]))],
                    size=need, replace=False
                )
                client_classes[cid].update(extras)

        # 分配样本
        client_indices = {cid: [] for cid in range(num_clients)}
        for lbl, idxs in buckets.items():
            owners = [cid for cid, cls in client_classes.items() if lbl in cls]
            splits = np.array_split(idxs, len(owners))
            for cid, part in zip(owners, splits):
                client_indices[cid].extend(part)
        client_indices = [np.array(v) for v in client_indices.values()]

    # ---------- 4) 构造 client_data，并加上共享数据子集 ----------
    global_label_hist = np.bincount(y, minlength=len(np.unique(y))) / len(y)
    stats = {'label_hist': {}, 'emd': {}, 'global_p': global_label_hist}

    client_data = {}
    for cid, idx in enumerate(client_indices):
        X_c, y_c = X_train[idx], y_train[idx]

        # 每个客户端抽取 α * |G| 的共享样本（可重复或不重复，这里不重复）
        if share_pct > 0 and share_fraction > 0:
            share_size = int(len(X_G) * share_fraction)
            share_idx  = rng.choice(len(X_G), size=share_size, replace=False)
            X_c = np.concatenate([X_c, X_G[share_idx]], axis=0)
            y_c = np.concatenate([y_c, y_G[share_idx]], axis=0)

        # 打乱
        perm = rng.permutation(len(X_c))
        X_c, y_c = X_c[perm], y_c[perm]

        client_data[cid] = (X_c, y_c)

        # --------- 统计分布 ----------
        hist = np.bincount(y_c, minlength=len(global_label_hist)) / len(y_c)
        stats['label_hist'][cid] = hist
        stats['emd'][cid] = _earth_movers_distance(hist, global_label_hist)

    # ---------- 5) 返回 ----------
    return client_data, (X_test, y_test), (X_val, y_val), stats


In [15]:
NUM_CLIENTS = 10
client_data, test_data, val_data, stats = partition_dataset(
    X_balanced, y_balanced,
    num_clients=NUM_CLIENTS,
    non_iid=False,
    classes_per_client=1,   # 最极端的 Non-IID
    share_pct=0.1,          # 从原始训练集中抽出 10% 做全局共享数据 G
    share_fraction=0.5      # 每个客户端收到其中 50% 的子集，即等于全局总数据的 5%
)

# 获取测试集
X_test, y_test = test_data

# 获取验证集
X_val, y_val = val_data

# 查看每个客户端样本数量
for i in range(NUM_CLIENTS):
    print(f"Client {i} size: {len(client_data[i][1])}")
print(f"Test set size: {len(X_test)}")


Client 0 size: 215918
Client 1 size: 215918
Client 2 size: 215918
Client 3 size: 215917
Client 4 size: 215917
Client 5 size: 215917
Client 6 size: 215917
Client 7 size: 215917
Client 8 size: 215917
Client 9 size: 215917
Test set size: 192784


In [16]:
# Label noise injection for robustness test (corrupt clients)
corrupt_clients = [0, 3, 4, 5]
for i in corrupt_clients:
    X, y = client_data[i]
    np.random.shuffle(y)
    client_data[i] = (X, y)

In [17]:
# 查看每个客户端样本数量
for i in range(NUM_CLIENTS):
    print(f"Client {i} size: {len(client_data[i][1])}")
print(f"Test set size: {len(X_test)}")

Client 0 size: 215918
Client 1 size: 215918
Client 2 size: 215918
Client 3 size: 215917
Client 4 size: 215917
Client 5 size: 215917
Client 6 size: 215917
Client 7 size: 215917
Client 8 size: 215917
Client 9 size: 215917
Test set size: 192784


In [18]:
from collections import Counter

for i in range(len(client_data)):
    _, labels = client_data[i]  # 直接取出标签数组
    label_count = Counter(labels.tolist())  # 转为 list 后计数
    print(f"Client {i} label distribution:")
    for label, count in sorted(label_count.items()):
        print(f"  Class {label}: {count}")
    print()


Client 0 label distribution:
  Class 0: 2667
  Class 1: 5351
  Class 2: 7614
  Class 3: 5658
  Class 4: 13536
  Class 5: 1102
  Class 6: 1086
  Class 7: 152871
  Class 8: 5577
  Class 9: 2260
  Class 10: 1081
  Class 11: 5686
  Class 12: 4138
  Class 13: 5618
  Class 14: 1673

Client 1 label distribution:
  Class 0: 2642
  Class 1: 5348
  Class 2: 7641
  Class 3: 5613
  Class 4: 13534
  Class 5: 1099
  Class 6: 1085
  Class 7: 152843
  Class 8: 5578
  Class 9: 2207
  Class 10: 1097
  Class 11: 5703
  Class 12: 4184
  Class 13: 5641
  Class 14: 1703

Client 2 label distribution:
  Class 0: 2667
  Class 1: 5422
  Class 2: 7598
  Class 3: 5635
  Class 4: 13569
  Class 5: 1102
  Class 6: 1109
  Class 7: 152963
  Class 8: 5557
  Class 9: 2230
  Class 10: 1064
  Class 11: 5629
  Class 12: 4136
  Class 13: 5567
  Class 14: 1670

Client 3 label distribution:
  Class 0: 2708
  Class 1: 5336
  Class 2: 7605
  Class 3: 5675
  Class 4: 13589
  Class 5: 1078
  Class 6: 1100
  Class 7: 152813
  Clas

In [19]:
pd.Series(y_val).value_counts()

7     136400
4      12157
2       6794
11      5082
3       5006
13      5002
8       4993
1       4855
12      3681
0       2403
9       1998
14      1506
10       969
6        969
5        969
Name: count, dtype: int64

In [20]:
pd.Series(y_test).value_counts()

7     136400
4      12157
2       6794
11      5083
3       5006
13      5003
8       4994
1       4854
12      3680
0       2402
9       1997
14      1507
5        969
6        969
10       969
Name: count, dtype: int64

In [21]:
label_encoder.classes_

array(['Backdoor', 'DDoS_HTTP', 'DDoS_ICMP', 'DDoS_TCP', 'DDoS_UDP',
       'Fingerprinting', 'MITM', 'Normal', 'Password', 'Port_Scanning',
       'Ransomware', 'SQL_injection', 'Uploading',
       'Vulnerability_scanner', 'XSS'], dtype=object)

In [22]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(y_balanced),
                                                 y=y_balanced)

class_weights = {k: v for k,v in enumerate(class_weights)}
class_weights

{0: 5.349310469213907,
 1: 2.6475472423643156,
 2: 1.8917342518043148,
 3: 2.567267255270132,
 4: 1.0572156369190104,
 5: 13.264788247841194,
 6: 13.264788247841194,
 7: 0.09422486934242817,
 8: 2.5738996922542876,
 9: 6.433525220670438,
 10: 13.264788247841194,
 11: 2.528676923884101,
 12: 3.491795944611985,
 13: 2.569114727008622,
 14: 8.530634098853932}

In [23]:
input_shape = X_test.shape[1:]

In [24]:
print(X_test.shape)
print(input_shape)

(192784, 91)
(91,)


In [25]:
num_classes = len(np.unique(y_test))
num_classes

15

# Federated Learning

In [26]:
# 创建模型架构
def create_model(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(90, activation='relu', kernel_regularizer=l2(0.01), input_shape=(input_dim,)))
    model.add(Dense(90, activation='relu', kernel_regularizer=l2(0.01)))
    model.add(Dense(num_classes, activation='softmax'))  # softmax 输出
    
    model.compile(optimizer=Adam(),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [27]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU is available:", gpus)
else:
    print("❌ No GPU found, running on CPU.")

✅ GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [28]:
# Configuration
MODEL_DIR = "aggregator-mpc-iid-wp-3cc-ws"
CLIENT_MODEL_DIR = "client_weight_mpc-iid-wp-3cc-ws"
GLOBAL_MODEL_ROUND_DIR = os.path.join(MODEL_DIR, "global_models_per_round")
os.makedirs(GLOBAL_MODEL_ROUND_DIR, exist_ok=True)
os.makedirs(CLIENT_MODEL_DIR, exist_ok=True)

In [29]:
# --- Hyperparameters ---
NUM_ROUNDS = 10
EPOCHS = 5
BATCH_SIZE = 256
NOISE_SCALE = 1e-3
TRIM_RATIO = 0.2

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    handlers=[
        logging.FileHandler('fl_training.log'),
        logging.StreamHandler(sys.stdout)  # ✅ 添加终端输出
    ]
)

logger = logging.getLogger(__name__)

# --- Federated Learning with Behavioral Validation ---

def client_validate_model(local_model, X_val_shared, y_val_shared, acc_threshold=0.8):
    """评估客户端模型在共享验证集上的表现，判断其是否合格"""
    _, acc = local_model.evaluate(X_val_shared, y_val_shared, verbose=0)
    return acc >= acc_threshold, acc

def generate_dynamic_validation_set(X_all, y_all, val_size=0.1, seed=None):
    """从全局训练集中按类别比例随机抽取一小部分作为共享验证集"""
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    for _, val_index in splitter.split(X_all, y_all):
        X_val_shared = X_all[val_index]
        y_val_shared = y_all[val_index]
    return X_val_shared, y_val_shared


# --- Noise mask generation ---
def generate_noise_masks(global_model, selected_indices):
    noise_masks = {}
    model_weights = global_model.get_weights()
    for idx in selected_indices:
        noise_masks[idx] = []
        for layer in model_weights:
            mask = np.random.normal(loc=0.0, scale=NOISE_SCALE, size=layer.shape)
            noise_masks[idx].append(mask)
    return noise_masks

# --- Save and evaluate model ---
def save_and_evaluate(global_model, global_history, round_num, X_test, y_test, label_encoder):
    round_model_path = os.path.join(GLOBAL_MODEL_ROUND_DIR, f"global_model_round_{round_num + 1}.h5")
    global_model.save(round_model_path)
    logger.info(f"\U0001F4BE Saved global model for round {round_num + 1}")

    if (round_num + 1) % 2 == 0:
        generate_reports(global_model, X_test, y_test, label_encoder, round_num)

# --- Generate classification reports ---
def generate_reports(model, X_test, y_test, label_encoder, round_num):
    predictions = model.predict(X_test)
    predicted_classes = np.argmax(predictions, axis=1)

    print("\n\U0001F4CB Classification Report:")
    print(classification_report(y_test, predicted_classes, 
                                target_names=label_encoder.classes_, digits=4))

    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(y_test, predicted_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, 
                yticklabels=label_encoder.classes_)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_round_{round_num + 1}.png")
    plt.close()

# --- Train selected clients with noise masks ---
def train_clients(global_model, client_data, selected_indices, noise_masks, total_samples,
                                   X_val, y_val, X_test, y_test, X_val_shared, y_val_shared, round_num,
                                   acc_threshold=0.5):
    masked_weights_sum = None
    effective_clients = []

    for idx in selected_indices:
        logger.info(f"\n📡 Training client {idx + 1}")
        print(f"\n📡 Training client {idx + 1}")

        local_model = tf.keras.models.clone_model(global_model)
        local_model.set_weights(global_model.get_weights())
        local_model.compile(optimizer=Adam(clipnorm=1.0), 
                            loss='sparse_categorical_crossentropy', 
                            metrics=['accuracy'])

        X_client, y_client = client_data[idx]

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
            ModelCheckpoint(
                os.path.join(CLIENT_MODEL_DIR, f"client_model_round_{round_num+1}_client_{idx}.h5"),
                monitor='val_loss', save_best_only=True, verbose=1)
        ]

        local_model.fit(
            X_client, y_client,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=1,
            callbacks=callbacks,
            class_weight=class_weights
        )

        # === 验证模型是否合格 ===
        passed, acc = client_validate_model(local_model, X_val_shared, y_val_shared, acc_threshold)
        if not passed:
            logger.warning(f"❌ Client {idx} excluded: val acc = {acc:.2f}")
            tf.keras.backend.clear_session()
            continue

        logger.info(f"✅ Client {idx} included: val acc = {acc:.2f}")
        effective_clients.append(idx)

        local_weights = local_model.get_weights()
        weight_factor = len(y_client) / total_samples
        masked_weights = [
            w * weight_factor + noise_masks[idx][i] 
            for i, w in enumerate(local_weights)
        ]

        if masked_weights_sum is None:
            masked_weights_sum = masked_weights
        else:
            masked_weights_sum = [
                sum_w + new_w 
                for sum_w, new_w in zip(masked_weights_sum, masked_weights)
            ]

        del local_model
        tf.keras.backend.clear_session()

    if not effective_clients:
        raise RuntimeError("❌ All clients excluded due to low validation accuracy.")
    
    print(f"🔎 Round {round_num+1}: {len(effective_clients)} clients passed validation")

    return masked_weights_sum

# --- Secure aggregation ---
def secure_aggregation(masked_weights_sum, noise_masks):
    total_noise = [np.zeros_like(w) for w in masked_weights_sum]
    for mask_list in noise_masks.values():
        for i in range(len(total_noise)):
            total_noise[i] += mask_list[i]

    for i, noise in enumerate(total_noise):
        if not np.allclose(noise, 0, atol=1e-6):
            logger.warning(f"Noise cancellation imperfect in layer {i}")

    return [
        masked_weights_sum[i] - total_noise[i] 
        for i in range(len(masked_weights_sum))
    ]

# --- Federated Training with Validation-Based Client Filtering ---
def federated_training(global_model, client_data, X_val, y_val, X_test, y_test, label_encoder, X_all, y_all):
    global_history = {'round': [], 'loss': [], 'accuracy': [], 'client_metrics': []}

    for round_num in range(NUM_ROUNDS):
        print(f"\n🔁 Federated Round {round_num + 1}/{NUM_ROUNDS}")

        # 🔄 每轮生成新的共享验证集
        X_val_shared, y_val_shared = generate_dynamic_validation_set(X_all, y_all, val_size=0.5, seed=round_num)

        selected_indices = list(client_data.keys())
        total_samples = sum(len(client_data[i][1]) for i in selected_indices)

        noise_masks = generate_noise_masks(global_model, selected_indices)

        masked_weights_sum = train_clients(
            global_model, client_data, selected_indices, noise_masks, total_samples,
            X_val, y_val, X_test, y_test, X_val_shared, y_val_shared,
            round_num, acc_threshold=0.5
        )

        aggregated_weights = secure_aggregation(masked_weights_sum, noise_masks)
        global_model.set_weights(aggregated_weights)
        print("✅ Global model updated.")

        save_and_evaluate(global_model, global_history, round_num, X_test, y_test, label_encoder)

    return global_model, global_history


# --- Main execution (example) ---
if __name__ == "__main__":
    # Initialize your global model
    global_model = create_model(X_test.shape[1], num_classes)

    trained_model, history = federated_training(
        global_model=global_model,
        client_data=client_data,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder,
        X_all=X_val,
        y_all=y_val
    )

    final_model_path = os.path.join(MODEL_DIR, "final_global_model.h5")
    trained_model.save(final_model_path)
    logger.info(f"\n💾 Final global model saved to {final_model_path}")



🔁 Federated Round 1/10
[2025-06-18 17:01:29,576] INFO: 
📡 Training client 1

📡 Training client 1
Epoch 1/5
840/844 [============================>.] - ETA: 0s - loss: 3.0077 - accuracy: 0.0333
Epoch 1: val_loss improved from inf to 2.71948, saving model to client_weight_mpc-iid-wp-3cc-ws\client_model_round_1_client_0.h5
844/844 [==============================] - 3s 3ms/step - loss: 3.0065 - accuracy: 0.0332 - val_loss: 2.7195 - val_accuracy: 0.0104
Epoch 2/5
839/844 [============================>.] - ETA: 0s - loss: 2.7088 - accuracy: 0.0137
Epoch 2: val_loss improved from 2.71948 to 2.70073, saving model to client_weight_mpc-iid-wp-3cc-ws\client_model_round_1_client_0.h5
844/844 [==============================] - 3s 3ms/step - loss: 2.7094 - accuracy: 0.0137 - val_loss: 2.7007 - val_accuracy: 0.0260
Epoch 3/5
832/844 [============================>.] - ETA: 0s - loss: 2.7049 - accuracy: 0.0488
Epoch 3: val_loss did not improve from 2.70073
844/844 [==============================] - 3s 

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔁 Federated Round 3/10
[2025-06-18 17:08:35,228] INFO: 
📡 Training client 1

📡 Training client 1
Epoch 1/5
830/844 [============================>.] - ETA: 0s - loss: 2.7355 - accuracy: 0.1207
Epoch 1: val_loss improved from inf to 2.80644, saving model to client_weight_mpc-iid-wp-3cc-ws\client_model_round_3_client_0.h5
844/844 [==============================] - 4s 5ms/step - loss: 2.7364 - accuracy: 0.1220 - val_loss: 2.8064 - val_accuracy: 8.6625e-04
Epoch 2/5
831/844 [============================>.] - ETA: 0s - loss: 2.7119 - accuracy: 0.0478
Epoch 2: val_loss improved from 2.80644 to 2.76467, saving model to client_weight_mpc-iid-wp-3cc-ws\client_model_round_3_client_0.h5
844/844 [==============================] - 3s 4ms/step - loss: 2.7137 - accuracy: 0.0472 - val_loss: 2.7647 - val_accuracy: 0.0055
Epoch 3/5
839/844 [============================>.] - ETA: 0s - loss: 2.7111 - accuracy: 0.0323
Epoch 3: val_loss improved from 2.76467 to 2.66643, saving model to client_weight_mpc-iid

In [30]:
# 确保测试集格式正确（true_classes 应为整数标签）
true_classes = y_test  # 如果是 one-hot，则使用 np.argmax(y_test, axis=1)

# 所有标签编号和名称
all_labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# 测试所有 global model
print("\n🌍 Testing Global Models:")
for fname in sorted(os.listdir(GLOBAL_MODEL_ROUND_DIR)):
    if fname.endswith(".h5"):
        model_path = os.path.join(GLOBAL_MODEL_ROUND_DIR, fname)
        model = load_model(model_path)
        pred_probs = model.predict(X_test)
        predicted_classes = np.argmax(pred_probs, axis=1)

        print(f"\n📋 Classification Report for Global Model {fname}:")
        print(classification_report(true_classes, predicted_classes,
                                    labels=all_labels,
                                    target_names=target_names,
                                    digits=4))

# 测试所有 client model
print("\n📡 Testing Client Models:")
for fname in sorted(os.listdir(CLIENT_MODEL_DIR)):
    if fname.endswith(".h5"):
        model_path = os.path.join(CLIENT_MODEL_DIR, fname)
        model = load_model(model_path)
        pred_probs = model.predict(X_test)
        predicted_classes = np.argmax(pred_probs, axis=1)

        print(f"\n📋 Classification Report for Client Model {fname}:")
        print(classification_report(true_classes, predicted_classes,
                                    labels=all_labels,
                                    target_names=target_names,
                                    digits=4))



🌍 Testing Global Models:
6025/6025 [==============================] - 8s 1ms/step

📋 Classification Report for Global Model global_model_round_1.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7689    0.9126    0.8346      2402
            DDoS_HTTP     0.9394    0.5849    0.7209      4854
            DDoS_ICMP     0.9841    0.8749    0.9263      6794
             DDoS_TCP     0.7377    0.5164    0.6075      5006
             DDoS_UDP     1.0000    0.9035    0.9493     12157
       Fingerprinting     0.1685    0.9247    0.2850       969
                 MITM     1.0000    0.9391    0.9686       969
               Normal     0.9995    1.0000    0.9997    136400
             Password     1.0000    0.1121    0.2017      4994
        Port_Scanning     0.2937    0.1793    0.2226      1997
           Ransomware     0.6409    0.2394    0.3486       969
        SQL_injection     0.4339    0.8991    0.5853      5083
            Uploading     0.56

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Global Model global_model_round_3.h5:
                       precision    recall  f1-score   support

             Backdoor     0.9642    0.3593    0.5235      2402
            DDoS_HTTP     0.9391    0.0381    0.0733      4854
            DDoS_ICMP     0.9684    0.9059    0.9361      6794
             DDoS_TCP     0.7520    0.4790    0.5852      5006
             DDoS_UDP     0.9999    0.8008    0.8893     12157
       Fingerprinting     0.1231    1.0000    0.2193       969
                 MITM     1.0000    0.9391    0.9686       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     1.0000    0.1121    0.2017      4994
        Port_Scanning     0.9831    0.0290    0.0564      1997
           Ransomware     0.3796    0.8813    0.5306       969
        SQL_injection     0.4344    0.9040    0.5868      5083
            Uploading     0.5656    0.4027    0.4705    

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Client Model client_model_round_10_client_1.h5:
                       precision    recall  f1-score   support

             Backdoor     0.8146    0.8560    0.8348      2402
            DDoS_HTTP     0.7592    0.8988    0.8231      4854
            DDoS_ICMP     0.9878    0.9573    0.9723      6794
             DDoS_TCP     0.9267    0.6464    0.7616      5006
             DDoS_UDP     0.9990    0.9802    0.9895     12157
       Fingerprinting     0.5379    0.8421    0.6565       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    0.9999    0.9999    136400
             Password     0.9125    0.1482    0.2550      4994
        Port_Scanning     0.4818    0.8793    0.6225      1997
           Ransomware     0.6754    0.4272    0.5234       969
        SQL_injection     0.4400    0.8886    0.5885      5083
            Uploading     0.6114    0.4451    

...

6025/6025 [==============================] - 7s 1ms/step

📋 Classification Report for Client Model client_model_round_9_client_4.h5:
                       precision    recall  f1-score   support

             Backdoor     0.0000    0.0000    0.0000      2402
            DDoS_HTTP     0.0000    0.0000    0.0000      4854
            DDoS_ICMP     0.0000    0.0000    0.0000      6794
             DDoS_TCP     0.0000    0.0000    0.0000      5006
             DDoS_UDP     0.0000    0.0000    0.0000     12157
       Fingerprinting     0.0051    0.9979    0.0101       969
                 MITM     0.3573    0.8514    0.5034       969
               Normal     0.0000    0.0000    0.0000    136400
             Password     0.0000    0.0000    0.0000      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.0000    0.0000    0.0000      5083
            Uploading     0.0000    0.0000    0

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 7s 1ms/step

📋 Classification Report for Client Model client_model_round_9_client_5.h5:
                       precision    recall  f1-score   support

             Backdoor     0.0000    0.0000    0.0000      2402
            DDoS_HTTP     0.0000    0.0000    0.0000      4854
            DDoS_ICMP     0.7664    0.7866    0.7763      6794
             DDoS_TCP     0.0000    0.0000    0.0000      5006
             DDoS_UDP     0.1437    0.9859    0.2508     12157
       Fingerprinting     0.0000    0.0000    0.0000       969
                 MITM     1.0000    0.1063    0.1922       969
               Normal     1.0000    0.6466    0.7854    136400
             Password     0.0000    0.0000    0.0000      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.0000    0.0000    0.0000      5083
            Uploading     0.0000    0.0000    0

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Client Model client_model_round_9_client_6.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7735    0.9042    0.8338      2402
            DDoS_HTTP     0.8000    0.7892    0.7946      4854
            DDoS_ICMP     0.9795    0.9720    0.9758      6794
             DDoS_TCP     1.0000    0.5585    0.7167      5006
             DDoS_UDP     0.9996    0.9794    0.9894     12157
       Fingerprinting     0.5620    0.8421    0.6741       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.5783    0.2974    0.3928      4994
        Port_Scanning     0.4518    0.9700    0.6165      1997
           Ransomware     0.7763    0.2436    0.3708       969
        SQL_injection     0.4501    0.7610    0.5657      5083
            Uploading     0.6194    0.4370    0